In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Load dataset from CSV
file_path = 'C:/Users/VICTUS/Downloads/Quote-Equity-ZOMATO-EQ-28-02-2024-to-28-02-2025.csv'
if not os.path.exists(file_path):
    raise FileNotFoundError(f"File not found: {file_path}. Please check the file path.")

# Load dataset
df = pd.read_csv(file_path)

# Ensure correct column naming
if 'Price' not in df.columns or 'Date' not in df.columns:
    raise ValueError(f"Missing expected columns: {df.columns}")

df.rename(columns={'Price': 'Closing Price'}, inplace=True)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Ensure Time Series column exists
df['Time Series'] = range(1, len(df) + 1)

# Moving Average (MA15)
df['MA15'] = df['Closing Price'].rolling(window=15, min_periods=1).mean()

# Adding Lag Features (Previous Closing Prices)
df['Closing Price Lag1'] = df['Closing Price'].shift(1)
df['Closing Price Lag2'] = df['Closing Price'].shift(2)
df['Closing Price Lag3'] = df['Closing Price'].shift(3)
df.dropna(inplace=True)  # Remove NaN values

# Features and Target
features = ['MA15', 'Closing Price Lag1', 'Closing Price Lag2', 'Closing Price Lag3']
target = 'Closing Price'

# Normalize Data
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

df[features] = scaler_x.fit_transform(df[features])
df[target] = scaler_y.fit_transform(df[[target]])

# Convert Data to LSTM Format (3D Shape)
X = df[features].values.reshape(df.shape[0], len(features), 1)  
y = df[target].values

# Train-Test Split
train_size = int(len(df) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Build LSTM Model
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(X_train.shape[1], 1)),
    LSTM(50, return_sequences=False),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

# Train the Model
model.fit(X_train, y_train, epochs=50, batch_size=16, verbose=1)

# Predict Future Prices
predicted_scaled = model.predict(X)
df['Predicted Price'] = scaler_y.inverse_transform(predicted_scaled)

# Inverse Transform the Actual Closing Price
df['Closing Price'] = scaler_y.inverse_transform(df[[target]])

# Compute Relative Strength Index (RSI)
def compute_rsi(data, window=14):
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window, min_periods=1).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window, min_periods=1).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

df['RSI'] = compute_rsi(df['Closing Price'])

# Compute Moving Average Convergence Divergence (MACD)
short_ema = df['Closing Price'].ewm(span=12, adjust=False).mean()
long_ema = df['Closing Price'].ewm(span=26, adjust=False).mean()
df['MACD'] = short_ema - long_ema
df['Signal Line'] = df['MACD'].ewm(span=9, adjust=False).mean()

# Compute Momentum Indicator
df['Momentum'] = df['Closing Price'] - df['Closing Price'].shift(5)

# Ensure no missing values before generating signals
df[['RSI', 'MACD', 'Predicted Price', 'Closing Price', 'Momentum']] = df[['RSI', 'MACD', 'Predicted Price', 'Closing Price', 'Momentum']].fillna(method='ffill')

# Generate trading signals using relaxed conditions
df['Signal'] = np.where(
    (df['RSI'] < 40) & (df['MACD'] > df['Signal Line']) & (df['Momentum'] > 0), 'Buy',
    np.where((df['RSI'] > 60) & (df['MACD'] < df['Signal Line']) & (df['Momentum'] < 0), 'Sell', 'Hold')
)

# Select necessary columns for visualization
trade_table_columns = ['Time Series', 'Closing Price', 'Predicted Price', 'RSI', 'MACD', 'Momentum', 'Signal']

# Show all rows
import pandas as pd
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns

# Print the trade table
# print(df[trade_table_columns].to_string())

# Filter out 'Hold' values
df_trade = df[['Date', 'Closing Price', 'Signal']].copy()
df_trade.rename(columns={'Closing Price': 'Price', 'Signal': 'Prediction'}, inplace=True)
df_trade = df_trade[df_trade['Prediction'] != 'Hold']
df_trade.reset_index(drop=True, inplace=True)

# Display Buy/Sell predictions
print(df_trade)


ValueError: Missing expected columns: Index(['Date ', 'series ', 'OPEN ', 'HIGH ', 'LOW ', 'PREV. CLOSE ', 'ltp ',
       'Price', 'vwap ', '52W H ', '52W L ', 'VOLUME ', 'VALUE ',
       'No of trades '],
      dtype='object')

In [3]:
# Plot Results
plt.figure(figsize=(12, 6))
plt.plot(df['Time Series'], df['Closing Price'], label='Actual Price', color='blue')
plt.plot(df['Time Series'], df['Predicted Price'], label='Predicted Price', linestyle='dashed', color='red')

# Mark Buy & Sell points
plt.scatter(df[df['Signal'] == 'Buy']['Time Series'], df[df['Signal'] == 'Buy']['Closing Price'], marker='^', color='green', label='Buy Signal')
plt.scatter(df[df['Signal'] == 'Sell']['Time Series'], df[df['Signal'] == 'Sell']['Closing Price'], marker='v', color='red', label='Sell Signal')

plt.xlabel("Time Series (Days)")
plt.ylabel("Stock Price")
plt.title("LSTM Stock Price Prediction with Buy/Sell Signals")
plt.legend()
plt.grid(True)
plt.show()


KeyError: 'Time Series'

<Figure size 1200x600 with 0 Axes>